# NB4 · Pipeline tự động & Dashboard
**FinReport Intelligence · Festival Chuyển đổi số 2026 · Bảng C · Case 3**

Notebook này làm 2 việc:
1. Chạy **`run_pipeline.py`**: toàn bộ 8 bước (S0 tiếp nhận → S7 xuất file nộp) bằng **một lệnh**, ghi kết quả lên Drive.
2. Mở **dashboard Streamlit** (`06_dashboard/app.py`) ngay trên Colab.

| File | Đặt ở Drive |
|---|---|
| `run_pipeline.py`, `requirements.txt` | `FinReport_Intelligence/` (thư mục gốc dự án) |
| `app.py` | `FinReport_Intelligence/06_dashboard/` |
| `.streamlit/config.toml` | `FinReport_Intelligence/.streamlit/` (không bắt buộc – giao diện vẫn chạy nếu thiếu) |
| 8 file `finreport_*.py` | `FinReport_Intelligence/lib/` |

## 1. Kết nối Drive & cài thư viện

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/FinReport_Intelligence"

import os
os.chdir(PROJECT)
# plotly 5.24 khớp với bản Plotly.js đi kèm Streamlit (plotly 6+/7 mã hóa mảng kiểu mới, biểu đồ sẽ hiển thị sai)
!pip install -q -r requirements.txt
import streamlit, plotly
print("streamlit", streamlit.__version__, "· plotly", plotly.__version__)

## 2. Chạy pipeline tự động (S0 → S7)

- Chế độ **đầy đủ** (~10–20 giây): gồm kiểm thử âm, 300 câu QA tự sinh, kiểm thử đột biến, độ nhạy ngưỡng.
- Chế độ **nhanh** (`--quick`, ~2 giây): bỏ các kiểm thử nặng – dùng khi chỉ cần làm mới kết quả.

Pipeline dừng ngay nếu một bước FAIL (vd dữ liệu bị sửa), không tạo file nộp từ dữ liệu chưa kiểm chứng.

In [ ]:
!python run_pipeline.py --project "{PROJECT}"

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{PROJECT}/03_outputs/pipeline_report.md", encoding="utf-8").read()))

## 3. Mở dashboard

**Cách A – trong Colab (nhanh nhất, chỉ bạn xem được):** chạy ô dưới, bấm vào đường link Colab hiện ra.

In [ ]:
import subprocess, time
subprocess.Popen(["streamlit", "run", "06_dashboard/app.py", "--server.port", "8501", "--server.headless", "true",
                  "--browser.gatherUsageStats", "false"], cwd=PROJECT,
                 stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(6)
print(open("/content/streamlit.log").read()[-400:])
from google.colab import output
output.serve_kernel_port_as_window(8501)

**Cách B – đường link công khai để giám khảo/đồng đội xem (Cloudflare Tunnel, không cần tài khoản):**
chạy sau Cách A (Streamlit đã chạy ở cổng 8501). Link có dạng `https://….trycloudflare.com`, tồn tại đến khi phiên Colab kết thúc.

In [ ]:
import subprocess, re, time, os
if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = proc.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("Dashboard công khai:", url or "chưa lấy được link – chạy lại ô này")

## 4. Dừng dashboard (khi xong)

In [ ]:
!pkill -f "streamlit run" ; pkill -f cloudflared ; echo "Đã dừng"

---
### Gợi ý demo trước giám khảo (5 phút)
1. **Sổ cái rủi ro** – đọc theo hàng: C012 xấu dần 0,03 → 0,93; C009 phục hồi 0,60 → 0,11.
2. **Thẻ cảnh báo** – chọn #1 (C012/2025): lý do bằng số, đóng góp risk score, bảng bằng chứng đọc thẳng từ báo cáo gốc, 32 luật kế toán PASS.
3. **What-if** – C003/2025: kéo *DSO mục tiêu* về 45 ngày → HIGH thành LOW; kịch bản vẫn đúng 32/32 luật kế toán.
4. **Trợ lý** – gõ "Doanh nghiệp nào có Debt/Assets cao nhất năm 2025?" → C005, kèm công thức & nguồn.
5. **Chất lượng mô hình** – KPI, so với baseline, độ nhạy ngưỡng. **Dữ liệu & nhật ký** – chuỗi run_id, SHA-256 file nộp.